In [1]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_nuclei_labels_output_dir,
    load_precomputed_nuclei_labels_if_available,
)
from utils.segmentation import predict_nuclei_labels

from utils.inference import predict_tiled_unet

import tifffile
import napari
import torch



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2026-04-18 03:55:56,770 [INFO] WRITING LOG OUTPUT TO C:\Users\adiezsanchez\.cellpose\run.log
2026-04-18 03:55:56,770 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0
2026-04-18 03:55:56,864 [INFO] ** TORCH CUDA version installed and working. **
2026-04-18 03:55:56,864 [INFO] ** TORCH CUDA version installed and working. **
2026-04-18 03:55:56,864 [INFO] >>>> using GPU (CUDA)
2026-04-18 03:55:58,184 [INFO] >>>> loading model C:\Users\adiezsanchez\.cellpose\models\cpsam


In [2]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

In [3]:
# Iterate through the .lif files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [4]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [5]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

2026-04-18 03:56:11,079 [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


[<Image layer 'edCerulean_CTRL' at 0x1bd719d6e60>,
 <Image layer 'edCitrine_FRET' at 0x1bd72299810>,
 <Image layer 'edCitrine_CTRL' at 0x1bd724df6a0>,
 <Image layer 'brightfield' at 0x1bd7284b970>]

In [6]:
import numpy as np
from skimage.filters import difference_of_gaussians

def _normalize_percentile(image: np.ndarray, pmin: int = 1, pmax: int = 99) -> np.ndarray:
    """
    Normalize an image to the [0,1] range based on percentile clipping.

    Args:
        image (np.ndarray): The input image to be normalized.
        pmin (int, optional): Lower percentile for normalization. Defaults to 1.
        pmax (int, optional): Upper percentile for normalization. Defaults to 99.

    Returns:
        np.ndarray: The normalized image with values in [0, 1].
    """
    vmin, vmax = np.percentile(image, (pmin, pmax))
    image = np.clip(image, vmin, vmax)  # clip outlier intensities
    if vmax > vmin:
        image = (image - vmin) / (vmax - vmin)  # rescale to [0, 1]
    else:
        image = image * 0  # avoid division by zero; returns zeros
    return image

def _normalize_full_range(image: np.ndarray) -> np.ndarray:
    """
    Normalize an image to the [0,1] range using full 0-100 percentiles.

    Args:
        image (np.ndarray): The input image to be normalized.

    Returns:
        np.ndarray: The normalized image.
    """
    return _normalize_percentile(image, pmin=0, pmax=100)

def simulate_fluo_from_bf(
    lif_image: np.ndarray, 
    markers: tuple, 
    low_sigma: float = 1, 
    high_sigma: float = 2
) -> np.ndarray:
    """
    Simulate a fluorescent cell boundary channel from a brightfield image.

    Args:
        lif_image (np.ndarray): 4D image data, assumed shape (C, Z, Y, X) or (C, ...).
        markers (tuple): Channel descriptors linking channel names to indices.
        low_sigma (float, optional): Lower sigma for DoG. Defaults to 1.
        high_sigma (float, optional): Higher sigma for DoG. Defaults to 2.

    Returns:
        np.ndarray: Simulated boundary image for UNet3D input.
    """
    # Find the brightfield channel index in the user-defined markers
    bf_index = None
    for marker in markers:
        if marker[0].lower() in ["brightfield", "bf"]:
            bf_index = marker[1]
            break
    if bf_index is None:
        print("Please define the brightfield channel index under MARKERS.")

    # Normalize to remove outliers (and help later UNet3D inference)  
    brightfield_norm = _normalize_percentile(lif_image[bf_index])

    # Remove out of focus brightfield haze using Difference of Gaussians (DoG)
    bf_dog = difference_of_gaussians(brightfield_norm, low_sigma, high_sigma)

    # Normalize DoG response to [0, 1] before inversion
    bf_dog = _normalize_full_range(bf_dog)

    # Invert black and white values to simulate fluorescently labelled cell boundaries
    bf_inv = 1 - bf_dog

    return bf_inv

sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x1bd72d32740>

In [7]:
# Predict root cell boundary probability maps using a pre-trained UNet3D model
root_pmaps = predict_tiled_unet(
    raw=sim_fluo_cell_walls,
    input_layout="ZYX",
    model_dir=MODEL_DIR,
    patch=(80, 160, 160),
    patch_halo=(8, 16, 16),
    stride_ratio=0.75,
    batch_size=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_amp=True,
)

# root_pmaps: (C_out, Z, Y, X)
viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

c:\Users\adiezsanchez\githubrepos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path,

<Image layer 'root_unet_pmap' at 0x1bd72c55a20>

In [8]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_nuclei_labels_output_dir(RAW_DATA_DIRECTORY, lif_container_id)
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_nuclei_labels_if_available(nuclei_labels_dir, lif_image_name)

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Add the prediction to the napari viewer
viewer.add_labels(nuclei_labels)

Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 3 ...loading


<Labels layer 'nuclei_labels' at 0x1bd870988b0>

In [ ]:
import numpy as np
from scipy.ndimage import binary_fill_holes
from skimage.morphology import binary_closing, binary_opening, binary_erosion, disk, ball
from skimage.measure import label

def generate_rough_root_3d(
    root_pmaps, 
    nuclei_labels, 
    probability_threshold=0.9, 
    visualize=False, 
    remove_nonconnected_components=False
):
    """
    Generate a coarse 3D root mask by combining predictions from a root boundary segmentation map 
    and nuclei label masks. This function includes slice-wise morphological closing and hole filling 
    to produce contiguous masks even when contours are broken.

    Optionally, non-root, non-connected components can be removed: if `remove_nonconnected_components=True`,
    after all morphological operations, the mask is 3D-labeled, and only the largest connected 3D component is 
    retained. All other components are removed. This ensures the mask corresponds to the principal connected structure.

    Args:
        root_pmaps (np.ndarray): Probability map of root boundaries (shape: [1, Z, Y, X]).
                                 The array should be indexed at [0] to obtain the 3D map.
        nuclei_labels (np.ndarray): 3D array of nuclei instance labels (same shape as ZYX).
        probability_threshold (float, optional): Threshold value for converting the root boundary probability map 
            to a binary mask. Voxels with a probability above this value are considered part of the root boundary.
            Default is 0.9. Higher values will result in a more conservative root boundary mask.
        visualize (bool, optional): If True, add relevant masks as layers to the global napari viewer. Default is False.
        remove_nonconnected_components (bool, optional): If True, retain only the largest 3D connected
            component of the mask. Default is False.

    Returns:
        np.ndarray: A boolean 3D mask representing the rough root segmentation. If 
            `remove_nonconnected_components=True`, only the largest connected component is retained.
    """
    # Threshold the root boundary probability map to create a binary mask
    root_boundary_mask = root_pmaps[0] > probability_threshold  # shape: (Z, Y, X)

    # Create a binary mask where nuclei are present
    nuclei_mask = nuclei_labels > 0  # shape: (Z, Y, X)

    # Combine the nuclei and root boundary masks as a starting point
    mask = (root_boundary_mask | nuclei_mask).astype(bool)

    # Preallocate the filled+closed output array
    filled_3d_closed = np.zeros_like(mask)

    # Perform slice-wise morphological closing and hole filling
    for z in range(mask.shape[0]):
        closed = binary_closing(mask[z], footprint=disk(5))           # Fill small holes/gaps in each slice
        filled_3d_closed[z] = binary_fill_holes(closed)               # Fill interior holes after closing

    if remove_nonconnected_components:
        # Label all 3D connected components
        labeled, num_labels = label(filled_3d_closed, return_num=True, connectivity=1)
        if num_labels > 0:
            # Retain only the largest 3D connected component
            counts = np.bincount(labeled.ravel())
            counts[0] = 0  # Ignore background
            largest_label = counts.argmax()
            filled_3d_closed = (labeled == largest_label)
        else:
            # No connected components found, return empty mask
            filled_3d_closed = np.zeros_like(filled_3d_closed, dtype=bool)

    # Optionally, visualize all relevant masks in napari
    if visualize:
        # Assumes that a global 'viewer' object exists
        viewer.add_image(root_boundary_mask, name="root_boundary_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(nuclei_mask, name="nuclei_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(mask, name="combined_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(filled_3d_closed, name="closed+filled_2d_slice", colormap="orange", blending="additive", opacity=0.6)

    return filled_3d_closed

In [ ]:
#TODO: Add logic to create folder to store root_3D_mask, check if it exists and load, else compute (merge with nuclei_labels_check logic)

In [27]:
# Generate a rough 3D root mask
filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)

In [ ]:
def fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, erosion=3, visualize=False):
    """
    Refine a preliminary 3D root mask by filling the root core based on slice-wise voxel occupancy,
    with additional erosion to ensure core robustness. The method preserves edges while solidifying the interior.

    Args:
        filled_3d_closed (np.ndarray): Boolean 3D array (Z, Y, X), preliminary root mask.
        occupancy_threshold (float, optional): Ratio (0-1). Voxels present in this fraction of slices belong to the core. Default is 0.9.
        erosion (int, optional): Radius of disk structuring element (in pixels) used for eroding the core 2D mask.
            Higher values result in a more conservative (smaller) core. Default is 3.
        visualize (bool, optional): Whether to add all relevant masks/layers to the global napari viewer.

    Returns:
        np.ndarray: Boolean 3D mask representing the refined and filled root mask (filled_final).
    """
    Z = filled_3d_closed.shape[0]

    # Compute per-pixel occupancy across Z
    occupancy = np.sum(filled_3d_closed, axis=0)  # shape: (Y, X)
    occupancy_norm = occupancy / Z

    # Threshold occupancy to define the 2D core present in sufficient slices
    threshold = int(np.ceil(occupancy_threshold * Z))
    core_2d = occupancy >= threshold

    # Erode the core to avoid overfilling edges
    core_2d_eroded = binary_erosion(core_2d, footprint=disk(erosion))

    #TODO: Limit expansion to the uppermost slice containing some % of core_2d_eroded and no empty background
    # Expand eroded core back to 3D
    core_3d = np.repeat(core_2d_eroded[np.newaxis, ...], Z, axis=0)

    # Combine with the original to fill the core while preserving original edges
    filled_final = filled_3d_closed | core_3d

    # Optional napari visualization
    if visualize:
        viewer.add_image(occupancy_norm, name="occupancy_map")
        viewer.add_image(core_2d, name="core_2d", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(core_2d_eroded, name="core_2d_eroded", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(filled_final, name="filled_root_3d", colormap="yellow", blending="additive", opacity=0.6)
        
    return filled_final


In [20]:
# Fill internal gaps inside rough 3D root mask
filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, visualize=True)

In [22]:
def smooth_outer_root_surface_3d(
    filled_root_3d,
    erosion=5,
    smoothing=3,
    visualize=False
):
    """
    Refine and smooth the outer surface of a 3D root mask by eroding the input mask 
    and then performing morphological closing and opening. 
    This procedure helps to remove small protrusions and irregularities on the surface,
    resulting in a smoother, more homogeneous root boundary.

    Args:
        filled_root_3d (np.ndarray): Boolean 3D mask (Z, Y, X) of the root structure.
        erosion (int, optional): Radius of ball structuring element for initial erosion. Larger values yield greater contraction of the root mask. Default is 5.
        smoothing (int, optional): Radius of ball structuring element applied to sequential closing and opening. Controls the degree of surface smoothing. Default is 3.
        visualize (bool, optional): If True, intermediate and final masks are rendered in the global napari viewer for inspection.

    Returns:
        np.ndarray: Boolean 3D mask representing the smoothed root (same shape as input).
    """
    # @root_segmentation.ipynb (13-2) -- Erode the root mask to eliminate thin surface structures and shrink mask slightly.
    eroded_root_3d = binary_erosion(filled_root_3d, footprint=ball(erosion))

    # @root_segmentation.ipynb (13-3,4) -- Smooth outer surface by closing small gaps, then remove small bulges with opening.
    smooth_mask = binary_closing(eroded_root_3d, ball(smoothing))
    smooth_mask = binary_opening(smooth_mask, ball(smoothing))

    # @root_segmentation.ipynb (13-6) -- Optionally visualize masks in napari viewer for user validation.
    if visualize:
        viewer.add_image(eroded_root_3d, name="eroded_root_3d", colormap="gray", blending="additive", opacity=0.8)
        viewer.add_image(smooth_mask, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

    return smooth_mask

In [23]:
# Smooth root outer surface to remove small protrusions
smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)

In [ ]:
#TODO: Move to feature_extraction.py

import numpy as np
from scipy.ndimage import distance_transform_edt
from skimage.measure import regionprops

def calculate_distance_to_root_surface(nuclei_labels, root_3d_mask, visualize=False):
    """
    Calculate a normalized distance map from the root surface and assign a depth value to each nucleus label, based on its centroid.
    The normalized distance is defined as the distance from each point inside the root mask to the nearest root surface, divided by the maximum thickness.
    Optionally, display the result in the global Napari viewer.

    Args:
        nuclei_labels (np.ndarray): 3D label array of nuclei (Z, Y, X).
        root_3d_mask (np.ndarray): Boolean 3D mask (Z, Y, X) of the root body.
        visualize (bool, optional): If True, display the normalized depth map in Napari.

    Returns:
        np.ndarray: 3D array (same shape as input) with normalized per-nucleus depth values; zero outside labeled regions.
    """
    # Pad the root mask with a border of zeros to ensure boundary proximity is detected at the edge of the image.
    mask_padded = np.pad(root_3d_mask.astype(bool), pad_width=1, mode='constant', constant_values=0)

    # Compute the distance transform on the padded mask.
    dist_padded = distance_transform_edt(mask_padded)

    # Remove padding to restore the shape to original spatial dimensions.
    dist_map = dist_padded[1:-1, 1:-1, 1:-1]

    # Normalize distances by the maximum value inside the root (aproximate root radius).
    max_dist = dist_map.max()
    if max_dist == 0:
        raise ValueError("Distance map is empty. Check mask.")

    dist_norm = dist_map / max_dist

    # Compute depth for each nucleus label based on the centroid coordinates in the normalized distance map.
    depth_per_label = {}
    props = regionprops(nuclei_labels)
    for prop in props:
        label_id = prop.label
        z, y, x = map(int, prop.centroid)  # Centroid coordinates
        depth = dist_norm[z, y, x]
        depth_per_label[label_id] = depth

    # Create an image where each voxel in a nucleus gets the assigned depth value for its label.
    depth_image = np.zeros_like(dist_map, dtype=float)
    for label_id, depth in depth_per_label.items():
        depth_image[nuclei_labels == label_id] = depth

    # Enhance contrast (clip the depth values between 1st and 99th percentile and re-normalize).
    nonzero = depth_image > 0
    if np.any(nonzero):
        p1, p99 = np.percentile(depth_image[nonzero], (1, 99))
        depth_image = np.clip(depth_image, p1, p99)
    depth_image = (depth_image - depth_image.min()) / (
        depth_image.max() - depth_image.min() + 1e-8
    )

    # Display the resulting normalized per-nucleus depth in Napari if requested.
    if visualize:
        viewer.add_image(
            depth_image,
            name="nuclei_depth_normalized",
            colormap="viridis",
            blending="additive"
        )

    return depth_image

In [ ]:
#TODO: Add logic to create folder to store depth map, check if it exists and load, else compute (merge with nuclei_labels_check logic)

# Calculate distance from each nuclei centroid to the root surface
# This will be used to approximate to which tissue layer each nucleus belongs
nuclei_depth_map = calculate_distance_to_root_surface(nuclei_labels, smooth_root_3d, visualize=True)